# 🔬 CV Image Processing Toolkit — Single-File Colab Edition

Classical computer vision with OpenCV, no deep learning. Everything from
[khalilurrrahmanridoykhan/cv-image-processing-toolkit](https://github.com/khalilurrrahmanridoykhan/cv-image-processing-toolkit)
in one notebook. Run **Runtime → Run all**.

**Contents**
1. Setup & sample data
2. The toolkit (I/O, preprocessing, edges, contours, shapes, segmentation, biomedical helpers)
3. Synthetic shape dataset + accuracy test
4. Real-photo pipeline (book, coins, stop sign, dice)
5. Health / biomedical demos (X-ray CLAHE, sickle cells, melanoma border, retina vessels, blood smear watershed)
6. 📤 Try it on **your own image**

> ⚠️ The health demos are educational image-processing exercises, not medical diagnostics.

## 1. Setup & sample data

In [ ]:
import os, sys, math, json, random, shutil, subprocess
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 100

# Sample photos live in the original repo; fetch them once (shallow clone, ~9 MB).
REPO_URL = "https://github.com/khalilurrrahmanridoykhan/cv-image-processing-toolkit.git"
ROOT = Path("/content/cv_data") if Path("/content").exists() else Path("cv_data")
REPO = ROOT / "repo"
if not (REPO / "data").exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

REAL   = REPO / "data" / "real"
HEALTH = REPO / "data" / "health"
SYNTH  = ROOT / "synthetic"          # generated below
print("OpenCV", cv2.__version__)
print("Real photos  :", sorted(p.name for p in REAL.glob("*.jpg")))
print("Health images:", sorted(p.name for p in HEALTH.glob("*.*") if p.suffix in (".jpg", ".png")))

## 2. The toolkit

In [ ]:
# ---------- I/O ----------
def load_image(image_path) -> np.ndarray:
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Image not found at path: {image_path}")
    return img

def to_rgb(img):       return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
def to_grayscale(img): return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


# ---------- Preprocessing ----------
def blur(gray, kernel_size=5):
    """Gaussian blur: smooths noise while preserving major boundaries."""
    if kernel_size % 2 == 0:
        raise ValueError(f"kernel_size must be an odd integer, got {kernel_size}")
    return cv2.GaussianBlur(gray, (kernel_size, kernel_size), 0)

def threshold_otsu(gray, invert=False):
    """Automatic binarization. invert=True -> dark objects become white."""
    flag = cv2.THRESH_BINARY_INV if invert else cv2.THRESH_BINARY
    _, binary = cv2.threshold(gray, 0, 255, flag + cv2.THRESH_OTSU)
    return binary

def apply_clahe(gray, clip_limit=2.0, tile_grid_size=8):
    """Contrast Limited Adaptive Histogram Equalization (standard for X-rays etc.)."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid_size, tile_grid_size))
    return clahe.apply(gray)

def morphological_clean(binary, kernel_size=3, iterations=1):
    """Opening (remove specks) then closing (fill holes)."""
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k, iterations=iterations)
    return cv2.morphologyEx(opened, cv2.MORPH_CLOSE, k, iterations=iterations)

def auto_canny(gray, sigma=0.33):
    """(Bonus) Canny with thresholds derived from the image median --
    addresses the repo's note that fixed (low, high) don't generalize across photos."""
    v = np.median(gray)
    return cv2.Canny(gray, int(max(0, (1 - sigma) * v)), int(min(255, (1 + sigma) * v)))


# ---------- Edges ----------
def sobel_edges(gray):
    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sx**2 + sy**2)
    return cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def canny_edges(gray, low=50, high=150):
    return cv2.Canny(gray, low, high)


# ---------- Contours ----------
def find_contours(binary, min_area=50.0, external_only=True):
    mode = cv2.RETR_EXTERNAL if external_only else cv2.RETR_LIST
    contours, _ = cv2.findContours(binary, mode, cv2.CHAIN_APPROX_SIMPLE)
    valid = [c for c in contours if cv2.contourArea(c) >= min_area]
    valid.sort(key=cv2.contourArea, reverse=True)
    return valid

def draw_contours(image, contours, color=(0, 255, 0), thickness=2):
    if image.ndim == 2 or image.shape[2] == 1:
        canvas = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    else:
        canvas = image.copy()
    cv2.drawContours(canvas, contours, -1, color, thickness)
    return canvas


# ---------- Geometric shape classification ----------
def classify_contour(contour):
    """circle / triangle / square / rectangle / pentagon / hexagon / polygon."""
    peri = cv2.arcLength(contour, True)
    if peri == 0:
        return "unknown"
    area = cv2.contourArea(contour)
    circularity = 4 * math.pi * area / (peri * peri)
    approx = cv2.approxPolyDP(contour, 0.035 * peri, True)
    n = len(approx)
    if n == 6 and circularity < 0.9:      # (added) hexagons are stable at 6 vertices; circles give ~8
        return "hexagon"
    if circularity > 0.82:
        return "circle"
    if n == 3:
        return "triangle"
    if n == 4:
        (_, _), (w, h), _ = cv2.minAreaRect(contour)
        if w == 0 or h == 0:
            return "rectangle"
        return "square" if max(w, h) / min(w, h) < 1.15 else "rectangle"
    if n == 5:
        return "pentagon"
    if circularity > 0.75:
        return "circle"
    return "polygon"

def _centroid(cnt):
    M = cv2.moments(cnt)
    if M["m00"] != 0:
        return int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
    return int(cnt[0][0][0]), int(cnt[0][0][1])

def annotate_shapes(image, contours, labels, color=(0, 255, 0), font_scale=0.6):
    canvas = image.copy()
    if canvas.ndim == 2 or canvas.shape[2] == 1:
        canvas = cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)
    for cnt, label in zip(contours, labels):
        cv2.drawContours(canvas, [cnt], -1, color, 2)
        cx, cy = _centroid(cnt)
        cv2.putText(canvas, label, (cx - 25, cy), cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale, (255, 0, 0), 2)
    return canvas


# ---------- Biomedical morphometry ----------
def classify_cell(contour):
    """normal_rbc / sickle_cell / irregular_cell from circularity + elongation."""
    peri = cv2.arcLength(contour, True)
    if peri == 0:
        return "unknown"
    area = cv2.contourArea(contour)
    circularity = 4 * math.pi * area / (peri * peri)
    (_, _), (w, h), _ = cv2.minAreaRect(contour)
    aspect = max(w, h) / min(w, h) if min(w, h) > 0 else 1.0
    if circularity >= 0.75 and aspect < 1.35:
        return "normal_rbc"
    if circularity < 0.68 and aspect >= 1.4:
        return "sickle_cell"
    return "irregular_cell"

def measure_lesion_irregularity(contour):
    """Area, perimeter, solidity (area / convex-hull area), circularity; flags solidity < 0.88."""
    area = cv2.contourArea(contour)
    peri = cv2.arcLength(contour, True)
    hull_area = cv2.contourArea(cv2.convexHull(contour))
    solidity = float(area) / hull_area if hull_area > 0 else 0.0
    circ = 4 * math.pi * area / (peri * peri) if peri > 0 else 0.0
    return {"area_pixels": float(area), "perimeter_pixels": float(peri),
            "solidity": round(solidity, 3), "circularity": round(circ, 3),
            "is_border_irregular": bool(solidity < 0.88)}

def annotate_cells(image, contours, cell_types):
    canvas = image.copy()
    if canvas.ndim == 2 or canvas.shape[2] == 1:
        canvas = cv2.cvtColor(canvas, cv2.COLOR_GRAY2RGB)
    colors = {"normal_rbc": (0, 255, 0), "sickle_cell": (255, 0, 0), "irregular_cell": (255, 200, 0)}
    for cnt, t in zip(contours, cell_types):
        cv2.drawContours(canvas, [cnt], -1, colors.get(t, (0, 255, 255)), 2)
    return canvas


# ---------- Segmentation (touching cells) ----------
def separate_touching_cells(binary, min_distance_factor=0.4, min_area=50.0):
    """Distance transform + watershed. Foreground must be white (255).
    Returns the mask with 1-px dividing lines (0) between formerly touching cells."""
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    clean = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k, iterations=2)
    sure_bg = cv2.dilate(clean, k, iterations=2)
    dist = cv2.distanceTransform(clean, cv2.DIST_L2, 5)
    if dist.max() == 0:
        return binary.copy()
    _, sure_fg = cv2.threshold(dist, min_distance_factor * dist.max(), 255, cv2.THRESH_BINARY)
    sure_fg = np.uint8(sure_fg)
    unknown = cv2.subtract(sure_bg, sure_fg)
    _, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    markers = cv2.watershed(cv2.cvtColor(clean, cv2.COLOR_GRAY2BGR), markers)
    out = clean.copy()
    out[markers == -1] = 0
    return out


# ---------- Small display helpers ----------
def show(images, titles=None, cmap_gray=True, figsize=None):
    """Show one or more images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize or (5.5 * n, 5.5))
    axes = np.atleast_1d(axes)
    for i, (ax, im) in enumerate(zip(axes, images)):
        ax.imshow(im, cmap="gray" if (im.ndim == 2 and cmap_gray) else None)
        if titles: ax.set_title(titles[i])
        ax.axis("off")
    plt.tight_layout(); plt.show()

print("Toolkit loaded ✅")

## 3. Synthetic shapes + accuracy test
Generates 30 images (6 each of circle, triangle, square, pentagon, rectangle) with exact ground truth,
then checks how well `classify_contour` does.

In [ ]:
CANVAS, SHAPE_COLOR, BG_COLOR = 200, (40, 40, 40), (255, 255, 255)

def _poly(center, radius, n, rot_deg):
    cx, cy = center
    a0 = math.radians(rot_deg - 90)
    return np.array([[cx + radius * math.cos(a0 + 2*math.pi*i/n),
                      cy + radius * math.sin(a0 + 2*math.pi*i/n)] for i in range(n)], dtype=np.int32)

def _rect(center, w, h, rot_deg):
    t = math.radians(rot_deg)
    R = np.array([[math.cos(t), -math.sin(t)], [math.sin(t), math.cos(t)]])
    corners = np.array([[-w/2, -h/2], [w/2, -h/2], [w/2, h/2], [-w/2, h/2]])
    return (corners @ R.T + np.array(center)).astype(np.int32)

def _center(margin):
    return random.randint(margin, CANVAS - margin), random.randint(margin, CANVAS - margin)

def make_shape(name):
    canvas = np.full((CANVAS, CANVAS, 3), BG_COLOR, dtype=np.uint8)
    if name == "circle":
        r = random.randint(25, 45); c = _center(r + 5)
        cv2.circle(canvas, c, r, SHAPE_COLOR, -1)
    elif name in ("triangle", "square", "pentagon"):
        n = {"triangle": 3, "square": 4, "pentagon": 5}[name]
        r = random.randint(30, 50); c = _center(r + 5)
        cv2.fillPoly(canvas, [_poly(c, r, n, random.randint(0, 359))], SHAPE_COLOR)
    elif name == "rectangle":
        w, h = random.randint(50, 80), random.randint(25, 45)
        c = _center(int(math.hypot(w, h) / 2) + 5)
        cv2.fillPoly(canvas, [_rect(c, w, h, random.randint(0, 359))], SHAPE_COLOR)
    return canvas

def generate_synthetic(per_shape=6, seed=42):
    random.seed(seed)
    SYNTH.mkdir(parents=True, exist_ok=True)
    truth = []
    for name in ["circle", "triangle", "square", "pentagon", "rectangle"]:
        for i in range(per_shape):
            fn = f"{name}_{len(truth):03d}.png"
            cv2.imwrite(str(SYNTH / fn), make_shape(name))
            truth.append({"filename": fn, "shape": name})
    (SYNTH / "ground_truth.json").write_text(json.dumps(truth, indent=2))
    return truth

truth = generate_synthetic()
print(f"Generated {len(truth)} synthetic images in {SYNTH}")

# --- accuracy test ---
results = []
for item in truth:
    img = load_image(SYNTH / item["filename"])
    bw = threshold_otsu(blur(to_grayscale(img)), invert=True)   # dark shape on white bg
    cnts = find_contours(bw, min_area=100)
    pred = classify_contour(cnts[0]) if cnts else "none"
    results.append((item["shape"], pred, item["filename"]))

acc = np.mean([t == p for t, p, _ in results])
print(f"\nOverall accuracy: {acc:.1%}  ({sum(t == p for t, p, _ in results)}/{len(results)})")

import collections
per = collections.defaultdict(lambda: [0, 0])
for t, p, _ in results:
    per[t][1] += 1; per[t][0] += (t == p)
for t, (ok, n) in per.items():
    print(f"  {t:<10} {ok}/{n}")
wrong = [(t, p, f) for t, p, f in results if t != p]
if wrong: print("Misclassified:", wrong)

# --- gallery ---
fig, axes = plt.subplots(5, 6, figsize=(14, 12))
for ax, (t, p, f) in zip(axes.ravel(), results):
    img = to_rgb(load_image(SYNTH / f))
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"{t} → {p}", color="green" if t == p else "red", fontsize=9)
plt.tight_layout(); plt.show()

## 4. Real-photo pipeline
**Read → grayscale → blur → Otsu threshold → edges → contours → shape label**, on `book.jpg`.

In [ ]:
img  = load_image(REAL / "book.jpg")
rgb, gray = to_rgb(img), to_grayscale(img)
blurred = blur(gray)
binary  = threshold_otsu(blurred)

show([rgb, gray, blurred, binary], ["Original", "Grayscale", "Gaussian blur", "Otsu threshold"], figsize=(20, 5))

In [ ]:
# Sobel vs Canny
show([blurred, sobel_edges(blurred), canny_edges(blurred), auto_canny(blurred)],
     ["gray + blur", "Sobel", "Canny (50, 150)", "Auto-Canny (median-based)"], figsize=(20, 5))

In [ ]:
# Contours + shape classification
contours = find_contours(binary, min_area=100)
labels = [classify_contour(c) for c in contours]
print(f"Contours found: {len(contours)}")
for i, l in enumerate(labels, 1):
    print(f"  Shape #{i}: {l}")

show([draw_contours(rgb, contours, thickness=3), annotate_shapes(rgb, contours, labels)],
     [f"{len(contours)} contour(s)", f"Classified: {labels}"], figsize=(14, 7))

### Same pipeline on every real photo
Notes from the original repo's findings: `coins.jpg` is a known hard case for a single global Otsu threshold
(reflective ridges make it zebra-striped), and `dice.jpg` is a cube seen at an angle, so its silhouette is a hexagon (labelled `hexagon` — a small addition to the original classifier). The stop sign is an octagon, but a circle and an octagon both simplify to ~8 vertices and have similar circularity, so this classifier calls it `circle` — a known limitation of purely geometric rules.

In [ ]:
photos = ["book.jpg", "stop_sign.jpg", "dice.jpg", "coins.jpg"]

def segment_object(im):
    """Binary mask with the object in white.
    Polarity rule: the image border is assumed to be background, so flip if the border is mostly white.
    Special case: the stop sign is picked out by colour (red in HSV), since street clutter defeats grayscale Otsu."""
    g = blur(to_grayscale(im))
    bw = threshold_otsu(g)
    border = np.concatenate([bw[0], bw[-1], bw[:, 0], bw[:, -1]])
    if border.mean() > 127:
        bw = 255 - bw
    return bw

def segment_red(im):
    hsv = cv2.cvtColor(im, cv2.COLOR_BGR2HSV)
    m = cv2.inRange(hsv, (0, 120, 70), (10, 255, 255)) | cv2.inRange(hsv, (170, 120, 70), (180, 255, 255))
    return morphological_clean(m, kernel_size=9, iterations=2)

fig, axes = plt.subplots(len(photos), 4, figsize=(18, 4.6 * len(photos)))
for row, name in zip(axes, photos):
    im = load_image(REAL / name); r = to_rgb(im)
    bw = segment_red(im) if name == "stop_sign.jpg" else segment_object(im)
    cnts = find_contours(bw, min_area=0.01 * im.shape[0] * im.shape[1])[:3]
    labs = [classify_contour(c) for c in cnts]
    row[0].imshow(r);                                         row[0].set_title(name)
    row[1].imshow(bw, cmap="gray");                           row[1].set_title("Mask (red-colour mask for stop sign)" if name == "stop_sign.jpg" else "Otsu mask")
    row[2].imshow(auto_canny(blur(to_grayscale(im))), cmap="gray"); row[2].set_title("Auto-Canny")
    row[3].imshow(annotate_shapes(r, cnts, labs, font_scale=1.2)); row[3].set_title(f"{labs}")
    for a in row: a.axis("off")
plt.tight_layout(); plt.show()

## 5. Health & biomedical demos
Classical techniques on open-licensed clinical images (sources/licenses in the original repo's `data/health/SOURCES.md`).

### 5.1 Radiology — chest X-ray CLAHE

In [ ]:
xray = to_grayscale(load_image(HEALTH / "chest_xray.png"))
show([xray, apply_clahe(xray, clip_limit=2.5, tile_grid_size=8)],
     ["Original grayscale", "CLAHE enhanced"], figsize=(14, 7))

### 5.2 Hematology — cell shape metrics (green = round, red = elongated/sickle-like, yellow = irregular)
> ℹ️ The sample image is an NHLBI teaching **illustration**, not a real smear, so the counts below demonstrate the circularity / aspect-ratio metrics rather than real detection. Try it on a real smear for meaningful results.

In [ ]:
im = load_image(HEALTH / "sickle_cell.jpg")
rgb_, gray_ = to_rgb(im), to_grayscale(im)
clean = morphological_clean(threshold_otsu(gray_, invert=True), kernel_size=3)
cnts = find_contours(clean, min_area=35)
types = [classify_cell(c) for c in cnts]
print(f"Total cells detected: {len(cnts)}")
for t in sorted(set(types)): print(f"  {t}: {types.count(t)}")
show([clean, annotate_cells(rgb_, cnts, types)], ["Cleaned binary mask", "Classified cells"], figsize=(14, 7))

### 5.3 Dermatology — melanoma border irregularity (solidity = area / convex-hull area)

In [ ]:
im = load_image(HEALTH / "skin_lesion_melanoma.jpg")
rgb_, gray_ = to_rgb(im), to_grayscale(im)
bw = threshold_otsu(blur(gray_, 15), invert=True)
lesions = find_contours(bw, min_area=1000)
main = lesions[0]
m = measure_lesion_irregularity(main)
print("Lesion morphometry:", m)

canvas = rgb_.copy()
cv2.drawContours(canvas, [cv2.convexHull(main)], -1, (255, 0, 0), 3)   # red  = convex hull
cv2.drawContours(canvas, [main], -1, (0, 255, 0), 3)                   # green = actual border
plt.figure(figsize=(9, 7)); plt.imshow(canvas); plt.axis("off")
plt.title(f"Border (green) vs convex hull (red) | solidity={m['solidity']} | irregular={m['is_border_irregular']}")
plt.show()

### 5.4 Ophthalmology — retinal edge map (green channel + CLAHE + Canny)
The result is noisy; a dedicated vessel filter (e.g. Frangi) would do better — a good next step.

In [ ]:
fundus = to_rgb(load_image(HEALTH / "retina_fundus.jpg"))
vessels = apply_clahe(fundus[:, :, 1], clip_limit=3.0, tile_grid_size=8)
edges = canny_edges(vessels, low=25, high=60)
show([fundus, vessels, edges], ["Original fundus", "Green channel + CLAHE", "Vessel edges"], figsize=(18, 6))

### 5.5 Microscopy — blood smear, distance transform + watershed
This photo was taken through a microscope eyepiece, so a plain global Otsu threshold latches onto the dark vignette instead of the cells
(the original repo's version hits exactly this). Fix: (1) detect the circular field of view and mask everything else,
(2) subtract the local background from the green channel — red cells absorb green light — and (3) threshold that.

In [ ]:
im = load_image(HEALTH / "blood_smear.jpg")
rgb_, gray_ = to_rgb(im), to_grayscale(im)

# 1. circular field of view (Hough on a downscaled copy)
s = 0.25
small = cv2.GaussianBlur(cv2.resize(gray_, None, fx=s, fy=s), (9, 9), 2)
circ = cv2.HoughCircles(small, cv2.HOUGH_GRADIENT, 1, 50, param1=100, param2=40, minRadius=60, maxRadius=120)
fov = np.zeros_like(gray_)
if circ is not None:
    x, y, r = circ[0][0] / s
    cv2.circle(fov, (int(x), int(y)), int(r * 0.88), 255, -1)   # 0.88 trims edge glare
else:
    fov[:] = 255                                                  # fallback: whole image

# 2. green channel minus local background -> cells show up bright
green = im[:, :, 1]
diff = cv2.subtract(cv2.GaussianBlur(green, (0, 0), 25), green)
diff = cv2.bitwise_and(diff, diff, mask=fov)

# 3. threshold + clean
smear_bin = threshold_otsu(cv2.GaussianBlur(diff, (3, 3), 0))
smear_bin = cv2.bitwise_and(smear_bin, fov)
smear_clean = morphological_clean(smear_bin, kernel_size=3)

# 4. split touching cells
smear_sep = separate_touching_cells(smear_clean, min_distance_factor=0.4)

before = find_contours(smear_clean, min_area=30)
after  = find_contours(smear_sep,   min_area=30)
print(f"Objects before watershed: {len(before)}")
print(f"Objects after watershed : {len(after)}")

overlay = draw_contours(rgb_, after, color=(0, 200, 0), thickness=1)
show([rgb_, smear_clean, smear_sep, overlay],
     ["Original smear", "Cell mask", "Watershed separated", "Detected cells"], figsize=(22, 7))

## 6. 📤 Try your own image
Run the next cell, pick an image, and tune the options in the form on the right.

In [ ]:
#@title Run the pipeline on your own image { run: "auto" }
use_upload = True        #@param {type:"boolean"}
dark_objects_on_light_bg = True  #@param {type:"boolean"}
blur_kernel = 5          #@param {type:"slider", min:1, max:31, step:2}
min_area = 200           #@param {type:"integer"}

def get_user_image():
    try:
        from google.colab import files
        up = files.upload()
        if up:
            name = next(iter(up)); return load_image(name), name
    except ImportError:
        print("Not in Colab -- using the sample dice photo.")
    return load_image(REAL / "dice.jpg"), "dice.jpg (sample)"

if use_upload:
    user_img, name = get_user_image()
else:
    user_img, name = load_image(REAL / "stop_sign.jpg"), "stop_sign.jpg (sample)"

# downscale huge photos for speed
h, w = user_img.shape[:2]
if max(h, w) > 1400:
    s = 1400 / max(h, w); user_img = cv2.resize(user_img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)

u_rgb, u_gray = to_rgb(user_img), to_grayscale(user_img)
u_blur = blur(u_gray, blur_kernel)
u_bin = threshold_otsu(u_blur, invert=dark_objects_on_light_bg)
u_cnt = find_contours(u_bin, min_area=min_area)
u_lab = [classify_contour(c) for c in u_cnt]
print(f"{name}: {len(u_cnt)} object(s) -> {u_lab}")

show([u_rgb, u_bin, auto_canny(u_blur), annotate_shapes(u_rgb, u_cnt, u_lab)],
     ["Original", "Otsu mask", "Auto-Canny", "Classified"], figsize=(22, 6))